In [ ]:
# If not installed yet, install pymovements
# needed in Binder, Google Colab, Deepnote
# (uncomment to install, then reload kernel)
#!pip install git+https://github.com/pymovements/pymovements.git@8d7fea442df4636343230d28335543445880530e
# Download example data file if not present
#!wget -q https://raw.githubusercontent.com/pymovements/interactive-demo/main/gaze-toy-example.csv

# Introduction to `pymovements`

- Processing eye movement data
- Open-source python package
- [Code](https://github.com/pymovements/pymovements)
- [Documentation](https://pymovements.readthedocs.io/en/latest/)
- [User Guide](https://pymovements.readthedocs.io/en/latest/user-guide/index.html)
- [Tutorials](https://pymovements.readthedocs.io/en/latest/tutorials/index.html)


## Navigation

- **0.** [Understanding Eye-Tracking Data](./00_understanding-data.ipynb)
- **1.** [Working with Raw Gaze Samples](./01_inspect-raw-samples.ipynb)
- **2.** [Detecting Oculomotoric Events](./02_event-detection.ipynb)
- **3.** [Downloading Public Datasets](./03_public-datasets.ipynb)

Bonus: [Plotting Summary](./10_plotting.ipynb)

_Acknowledgement: This presentation builds on work of several `pymovements` authors._

---

# 1. Working with Raw Gaze Samples

Goal:
 - working with gaze samples,
 - visualizing them,
 - and transforming them.

In [ ]:
# Import `pymovements` as the alias `pm` for convenience
import pymovements as pm
from pymovements.gaze.experiment import Experiment

import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'svg'

Load gaze data. They are accessible as time-ordered raw samples in `gaze.samples`. 

Example data: [gaze-toy-example.csv](./gaze-toy-example.csv)

Three columns:
- time [ms],
- x [px],
- y [px]

In [ ]:
# Define Experiment
experiment = Experiment(
    screen_width_px=1280,
    screen_height_px=1024,
    screen_width_cm=38,
    screen_height_cm=30.2,
    distance_cm=68,
    origin="upper left",  # orientation of coordinates
    sampling_rate=250.0,  # recording sampling rate
)

These values are relevant for transformations and calculating measures later.

In [ ]:
# Load example data from csv
gaze = pm.gaze.from_csv(
    "./gaze-toy-example.csv",  # file location
    experiment=experiment,
    time_column="time",  # specify time column
    pixel_columns=["x", "y"],  # specify pixel columns
)

Let's look at the first few samples.

In [ ]:
gaze.samples.head(5)

Each row corresponds to one gaze sample. Timestamps are listed in the ``time`` column, and horizontal and vertical gaze positions in pixel coordinates can be found in the `pixel` column. Depending on the loader and input format, additional channels such as binocular coordinates or quality measures may also be present.

## 1.1 Inspecting Raw Samples with Plots

Visual inspection is an essential first step when working with newly loaded gaze data.
Time-series plots **help reveal problems** like signal loss, noise, blinks, sampling irregularities, or calibration problems before any preprocessing is applied.

Using the `pymovements.plotting.traceplot` function, we can visualize raw gaze samples from a `pymovements.Gaze` object.

In [ ]:
pm.plotting.traceplot(gaze)

The plot shows the 'continuous' trajectory of gaze positions across the stimulus, allowing inspection of spatial gaze behavior.

To get a **clearer impression of the time**:

We can examine each recorded signal separately using `pymovements.plotting.tsplot`. The x-axis represents time, as defined by the gaze sample timestamps. In this example, we plot the `x` and `y` channels. 

In [ ]:
gaze_unnested = gaze.clone()
gaze_unnested.unnest("pixel")
gaze_unnested

In [ ]:
pm.plotting.tsplot(
    gaze_unnested,
    xlabel="time [ms]",
    channels=["pixel_x", "pixel_y"],
    share_y=False,
    line_color="darkblue",
    n_rows=2,
    n_cols=1,
    zero_centered_yaxis=False,
)

You can easily guess the type of data:

A text with 6 lines. The `pixel_x` column hints the fixations.

**NOTE:** When using these plots, for a report or project, please remember to add the units into the label.

## 1.2 Data Loss

While one can visually inspect the gaze, it is not an objective measure.

Data Loss is the first data quality metric one should look at.

Are there NaNs in the data?

In [ ]:
gaze.measure_samples(
    "data_loss",
    column="pixel",
    sampling_rate=gaze.experiment.sampling_rate,
    # unit="ratio",
)

[gaze-toy-example.csv](./gaze-toy-example.csv)

What unit are we speaking?

- by total samples / count
- time

Other data quality measures can be calculated. See later in [2. Detecting Oculomotoric Events](./02_event-detection.ipynb).

## 1.3 Transforming Raw Samples

Raw pixel coordinates are tied to a **specific screen setup** and **viewing distance**.

Alternative representations are better suited for meaningful interpretation. These transformations operate directly on the raw samples and rely on the `Experiment`'s metadata defined earlier.

### 1.3.1 `pix2deg()`: From Pixels to Degrees of Visual Angle

Gaze is typically in screen pixels:
- Useful for display-based inspection,
- but not comparable across setups (pixel units depend on screen size and viewing distance)

| Coordinate System | Emphasis | Columns | Units |
|:---|:---|:---:|:---:|
| Screen coordinates | Where does one look at **on the screen**. | `pixel_x`, `pixel_y` | [$\mathrm{px}$], [$\mathrm{px}$] \| [$\mathrm{cm}$], [$\mathrm{cm}$] |
| Degrees of visual angle | In what **direction** does the **eye itself** look. | `position_x`, `position_y` | [$\deg$], [$\deg$] |

<!-- <a href="https://developer.tobii.com/xr/learn/eye-behavior/visual-angles/"><img src="./img/field-of-view.jpeg" alt="Visualisation of 2D Degrees of Visual Angle" width="500" height="600"> </a>
<small><a href="https://developer.tobii.com/xr/learn/eye-behavior/visual-angles/">(c) Tobii</a></small> -->
![Visualisation of 2D Degrees of Visual Angle](./img/field-of-view.jpeg)<small><a href="https://developer.tobii.com/xr/learn/eye-behavior/visual-angles/">(c) Tobii</a></small>

Degrees of visual angle reflect the actual angular displacement of the eye relative to the observer's viewpoint and are therefore **comparable across different screen setups and viewing distances**.

The `pymovements.gaze.transforms.pix2deg` converts pixel coordinates into degrees of visual angle (dva) using the experiment's screen geometry and viewing distance.

Requirements:
- A pixel-based gaze column must be available (by default named "pixel")
- A `pymovements.Experiment` must be attached, because screen size and distance are needed for the conversion

In [ ]:
gaze.pix2deg()

In [ ]:
gaze

In [ ]:
gaze_unnested = gaze.clone()
gaze_unnested.unnest("position")

pm.plotting.tsplot(
    gaze_unnested,
    xlabel="time [ms]",
    channels=["position_x", "position_y"],
    share_y=False,
    line_color="darkblue",
    n_rows=2,
    n_cols=1,
    zero_centered_yaxis=False,
)

The plot above, might look the same, but the change implies different units and is not dependent on the recording setup any more.

### 1.3.2 `pos2vel()`: From Position to Velocity

Velocity is computed from changes in gaze position over time and is **central to event detection algorithms for saccades and fixations**.


In pymovements, velocity is computed explicitly from position data with the `pymovements.gaze.transforms.pos2vel` function, using the sampling rate stored in the `Experiment`.

In [ ]:
gaze.pos2vel()

In [ ]:
gaze

Let's look at these new columns with `tsplot`.

In [ ]:
gaze_unnested = gaze.clone()
gaze_unnested.unnest("velocity")

pm.plotting.tsplot(
    gaze_unnested,
    xlabel="time [ms]",
    channels=["velocity_x", "velocity_y"],
    share_y=False,
    line_color="darkblue",
    n_rows=2,
    n_cols=1,
    zero_centered_yaxis=False,
)

plt.show()

The following plot illustrates velocity, i.e. how quickly the eye moves at each time point. Periods of relative stability (low velocity) typically correspond to fixations, whereas sharp peaks in the signal indicate rapid eye movements such as saccades.

Find more details about these methods in the tutorials at [Preprocessing Raw Gaze Data](https://pymovements.readthedocs.io/en/latest/tutorials/preprocessing-raw-data.html).

---

Next: [2. Detecting Oculomotoric Events](./02_event-detection.ipynb)